# 01 · Model comparison

**Question:** which Bedrock model should run extract and triage?

Each run uses the full Graph on the dev split. Calls are cached in `eval/cache/`,
so only the first run of each model costs money (about $0.03 for Nova Lite).

**Credits:** Nova models are billed as Bedrock and are usually covered by AWS
credits. Claude on Bedrock is billed through AWS Marketplace and may not be;
the Claude cell is off by default.

In [ ]:
import lab

## Candidates
Prices are USD per million tokens from the Bedrock pricing page; check your region.

In [ ]:
CANDIDATES = {
    "nova-micro": dict(
        fast_model_id="apac.amazon.nova-micro-v1:0",
        fast_model_input_price=0.035,
        fast_model_output_price=0.14,
    ),
    "nova-lite": dict(
        fast_model_id="apac.amazon.nova-lite-v1:0",
        fast_model_input_price=0.06,
        fast_model_output_price=0.24,
    ),
}
INCLUDE_CLAUDE = False  # billed via AWS Marketplace; may not be covered by credits
if INCLUDE_CLAUDE:
    CANDIDATES["claude-haiku-4.5"] = dict(
        fast_model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        fast_model_input_price=1.00,
        fast_model_output_price=5.00,
    )

In [ ]:
runs = {}
for label, overrides in CANDIDATES.items():
    run_settings = lab.settings(**overrides)
    runs[label] = lab.run(lab.build_deps(run_settings), run_settings)
    print(label, "done,", len(runs[label].failed), "failed")

## Results
Critical recall comes first: a model that misses critical cases doesn't ship, whatever else it wins.

In [ ]:
lab.compare(runs)

## Where they disagree

In [ ]:
if {"nova-micro", "nova-lite"} <= runs.keys():
    display(lab.priority_changes(runs["nova-micro"], runs["nova-lite"]))

## Decision
Write the chosen model and why here, then promote it (see `notebooks/README.md`).